# Paper Acceptance Prediction with Fine-tuned Model

This notebook demonstrates how to use a fine-tuned language model from Hugging Face to predict paper acceptance.

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install transformers torch accelerate tqdm -q

# Check GPU availability
import torch
if torch.cuda.is_available():
    print(f"GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("No GPU available, using CPU")

## 2. Download Inference Script

In [ ]:
# Download the inference script
!wget https://raw.githubusercontent.com/your-repo/inference_colab.py

# Or copy-paste the script content
# with open('inference_colab.py', 'w') as f:
#     f.write('''[paste script content here]''')

## 3. Initialize the Predictor

In [ ]:
from inference_colab import PaperAcceptancePredictor

# Replace with your Hugging Face model name
MODEL_NAME = "your-username/your-model-name"

# For private models, add your Hugging Face token
# HF_TOKEN = "hf_..."
# predictor = PaperAcceptancePredictor(MODEL_NAME, use_auth_token=HF_TOKEN)

# Initialize predictor
predictor = PaperAcceptancePredictor(MODEL_NAME)

## 4. Single Paper Prediction

In [ ]:
# Example paper text
paper_text = """
Abstract:
This paper presents a novel approach to computer vision using transformer architectures.
We propose a new method that combines self-attention mechanisms with convolutional layers
to achieve state-of-the-art performance on image classification tasks.

Introduction:
In recent years, deep learning has revolutionized the field of computer vision.
Traditional convolutional neural networks (CNNs) have been the dominant architecture
for image-related tasks. However, the recent success of transformers in natural language
processing has inspired researchers to explore their potential in vision tasks...
"""

# Make prediction
result = predictor.predict(paper_text, return_probabilities=True)

# Display results
print("="*50)
print(f"Decision: {result['decision'].upper()}")
print(f"Confidence: {result['confidence']:.2%}")
print(f"Accept Probability: {result['accept_probability']:.2%}")
print(f"Reject Probability: {result['reject_probability']:.2%}")
print("="*50)

## 5. Batch Prediction

In [ ]:
# Multiple papers
papers = [
    "Paper 1: Abstract and introduction text...",
    "Paper 2: Abstract and introduction text...",
    "Paper 3: Abstract and introduction text...",
]

# Batch prediction
results = predictor.predict(papers, batch_size=2, return_probabilities=True)

# Display results
for i, result in enumerate(results, 1):
    print(f"Paper {i}: {result['decision']} (confidence: {result['confidence']:.2%})")

## 6. Interactive Demo

In [ ]:
# Run interactive demo
predictor.interactive_demo()

## 7. Upload and Process Your Own Papers

In [ ]:
# Upload files from your computer
from google.colab import files
import json

uploaded = files.upload()

# Process uploaded JSON file
for filename in uploaded.keys():
    if filename.endswith('.json'):
        # Load papers from JSON
        with open(filename, 'r') as f:
            papers_data = json.load(f)
        
        # Extract texts (adjust based on your JSON structure)
        if isinstance(papers_data, list):
            texts = [p['text'] if isinstance(p, dict) else p for p in papers_data]
        else:
            texts = list(papers_data.values())
        
        # Make predictions
        results = predictor.predict(texts, return_probabilities=True)
        
        # Save results
        output_filename = f"predictions_{filename}"
        with open(output_filename, 'w') as f:
            json.dump(results, f, indent=2)
        
        print(f"Processed {len(results)} papers from {filename}")
        print(f"Results saved to {output_filename}")
        
        # Download results
        files.download(output_filename)

## 8. Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming you have results from batch prediction
if 'results' in locals() and len(results) > 0:
    # Extract data
    decisions = [r['decision'] for r in results]
    confidences = [r['confidence'] for r in results]
    accept_probs = [r['accept_probability'] for r in results]
    
    # Create visualizations
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Decision distribution
    ax1 = axes[0]
    accept_count = decisions.count('accept')
    reject_count = decisions.count('reject')
    ax1.bar(['Accept', 'Reject'], [accept_count, reject_count], color=['green', 'red'])
    ax1.set_title('Decision Distribution')
    ax1.set_ylabel('Count')
    
    # Confidence distribution
    ax2 = axes[1]
    ax2.hist(confidences, bins=20, edgecolor='black')
    ax2.set_title('Confidence Distribution')
    ax2.set_xlabel('Confidence')
    ax2.set_ylabel('Count')
    
    # Accept probability distribution
    ax3 = axes[2]
    ax3.hist(accept_probs, bins=20, edgecolor='black', color='blue')
    ax3.axvline(x=0.5, color='red', linestyle='--', label='Decision Threshold')
    ax3.set_title('Accept Probability Distribution')
    ax3.set_xlabel('Accept Probability')
    ax3.set_ylabel('Count')
    ax3.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print("\nSummary Statistics:")
    print(f"Total Papers: {len(results)}")
    print(f"Accepted: {accept_count} ({accept_count/len(results)*100:.1f}%)")
    print(f"Rejected: {reject_count} ({reject_count/len(results)*100:.1f}%)")
    print(f"Average Confidence: {np.mean(confidences):.2%}")
    print(f"Min/Max Confidence: {np.min(confidences):.2%} / {np.max(confidences):.2%}")
else:
    print("No results to visualize. Run batch prediction first.")